In [1]:
!pip install -q transformers timm accelerate opencv-python

In [5]:
!unzip -q "/content/New_Test_Dataset.zip" -d ./New_Test_Dataset

In [3]:
!unzip -q "/content/Random_Train_51.zip" -d ./Random_Train_51

In [6]:
import os
import json
import cv2
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from transformers import DetrImageProcessor, DetrForObjectDetection, TrainingArguments, Trainer

# ==============================================================================
# --- CONFIGURATION PATHS ---
# ==============================================================================
TRAIN_IMG_DIR   = "/content/Random_Train_51/Random_Train_51/images"
TRAIN_MASK_DIR  = "/content/Random_Train_51/Random_Train_51/masks"
TRAIN_LABELS_JSON = "/content/Random_Train_51/Random_Train_51/labels.json"

TEST_IMG_DIR    = "/content/New_Test_Dataset/New_Test_Dataset/images"
TEST_MASK_DIR   = "/content/New_Test_Dataset/New_Test_Dataset/masks"
TEST_LABELS_JSON = "/content/New_Test_Dataset/New_Test_Dataset/labels.json"

MODEL_NAME      = "facebook/detr-resnet-50"
OUTPUT_DIR      = "./detr_dynamic_finetuned"
BATCH_SIZE      = 4
EPOCHS          = 75
CONF_THRESHOLD  = 0.1
IOU_THRESHOLD   = 0.5

NUM_LABELS = 10
# ==============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

categories_mapping = {
    "artefatto":         1,
    "emazia":            2,
    "eosinophil":        3,
    "epithelial":        4,
    "epithelial ciliated": 5,
    "lymphocyte":        6,
    "mast cell":         7,
    "metaplastic":       8,
    "muciparous":        9,
    "neutrophil":        10,
}

processor = DetrImageProcessor.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------------------------
# 1. RUNTIME COORDINATE EXTRACTIONS
# ------------------------------------------------------------------------------
def get_coco_bbox(mask_path):
    """Returns COCO [xmin, ymin, width, height] format for the processor."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    y_idx, x_idx = np.where(mask > 0)
    if len(x_idx) == 0 or len(y_idx) == 0:
        return None
    xmin, xmax = float(np.min(x_idx)), float(np.max(x_idx))
    ymin, ymax = float(np.min(y_idx)), float(np.max(y_idx))
    w = xmax - xmin
    h = ymax - ymin
    if w <= 0 or h <= 0:
        return None
    return [xmin, ymin, w, h]


def get_absolute_bbox(mask_path):
    """Returns absolute corner coordinates [xmin, ymin, xmax, ymax] for IoU matching."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    y_idx, x_idx = np.where(mask > 0)
    if len(x_idx) == 0 or len(y_idx) == 0:
        return None
    return [float(np.min(x_idx)), float(np.min(y_idx)),
            float(np.max(x_idx)), float(np.max(y_idx))]


# ------------------------------------------------------------------------------
# 2. CUSTOM DATASET PIPELINE
# ------------------------------------------------------------------------------
class DynamicMaskDETRDataset(Dataset):
    def __init__(self, json_path, img_dir, mask_dir, processor):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.processor = processor

        with open(json_path, 'r') as f:
            data_content = json.load(f)

        self.entries = []
        for img_entry in data_content["images"]:
            img_path = os.path.join(self.img_dir, img_entry["image_name"])
            if not os.path.exists(img_path):
                continue
            # Check that at least one mask produces a valid bbox
            has_valid = False
            for mask_entry in img_entry["labels"]:
                mask_path = os.path.join(self.mask_dir, mask_entry["mask_file"])
                if (mask_entry["label"] in categories_mapping
                        and os.path.exists(mask_path)
                        and get_coco_bbox(mask_path) is not None):
                    has_valid = True
                    break
            if has_valid:
                self.entries.append(img_entry)

        print(f"Loaded {len(self.entries)} valid samples from {json_path}")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry    = self.entries[idx]
        img_path = os.path.join(self.img_dir, entry["image_name"])
        image    = Image.open(img_path).convert("RGB")

        annotations = []
        for mask_entry in entry["labels"]:
            mask_path  = os.path.join(self.mask_dir, mask_entry["mask_file"])
            label_text = mask_entry["label"]
            if not os.path.exists(mask_path) or label_text not in categories_mapping:
                continue
            coco_box = get_coco_bbox(mask_path)
            if coco_box is None:
                continue
            annotations.append({
                "image_id":   idx,
                "category_id": categories_mapping[label_text],
                "bbox":       coco_box,
                "area":       coco_box[2] * coco_box[3],
                "iscrowd":    0,
            })

        inputs = self.processor(
            images=image,
            annotations={"image_id": idx, "annotations": annotations},
            return_tensors="pt",
        )
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels":       inputs["labels"][0],
        }


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    pixel_values = [item["pixel_values"] for item in batch]
    encoding     = processor.pad(pixel_values, return_tensors="pt")
    labels       = [item["labels"] for item in batch]
    batch_dict   = {"pixel_values": encoding["pixel_values"]}
    if "pixel_mask" in encoding:
        batch_dict["pixel_mask"] = encoding["pixel_mask"]
    batch_dict["labels"] = labels
    return batch_dict


train_dataset = DynamicMaskDETRDataset(
    TRAIN_LABELS_JSON, TRAIN_IMG_DIR, TRAIN_MASK_DIR, processor
)

# ------------------------------------------------------------------------------
# 3. MODEL TRAINING
# ------------------------------------------------------------------------------
model = DetrForObjectDetection.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=1e-5,
    weight_decay=1e-4,
    save_strategy="epoch",
    logging_steps=10,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
)

print("\n--- STARTING DETR FINE-TUNING ---")
trainer.train()

model.eval()
model.to(device)

# ------------------------------------------------------------------------------
# 4. BOX TRANSFORMATION & EVALUATION
# ------------------------------------------------------------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    areaA = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    areaB = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    return interArea / float(areaA + areaB - interArea)

def center_to_corners(cx, cy, w, h, img_w, img_h):
    xmin = (cx - 0.5 * w) * img_w;  ymin = (cy - 0.5 * h) * img_h
    xmax = (cx + 0.5 * w) * img_w;  ymax = (cy + 0.5 * h) * img_h
    return [xmin, ymin, xmax, ymax]

with open(TEST_LABELS_JSON, 'r') as f:
    test_labels_data = json.load(f)

# Reverse map: int ID → class name, for readable output
id_to_class = {v: k for k, v in categories_mapping.items()}

# Per-class counters
class_gt_counts  = {cid: 0 for cid in categories_mapping.values()}
class_hits       = {cid: 0 for cid in categories_mapping.values()}
class_iou_sum    = {cid: 0.0 for cid in categories_mapping.values()}

total_gt_objects   = 0
correct_detections = 0
running_iou        = 0.0

print("\n--- RUNNING EVALUATION ON TEST SPLIT ---")
for img_entry in test_labels_data["images"]:
    img_path = os.path.join(TEST_IMG_DIR, img_entry["image_name"])
    if not os.path.exists(img_path):
        continue

    gt_boxes = []
    for mask_entry in img_entry["labels"]:
        mask_path  = os.path.join(TEST_MASK_DIR, mask_entry["mask_file"])
        label_text = mask_entry["label"]
        if label_text not in categories_mapping:
            continue
        bbox = get_absolute_bbox(mask_path)
        if bbox is not None:
            cid = categories_mapping[label_text]
            gt_boxes.append({"box": bbox, "label": cid})
            class_gt_counts[cid] += 1  # count GT per class

    if not gt_boxes:
        continue
    total_gt_objects += len(gt_boxes)

    image        = Image.open(img_path).convert("RGB")
    img_w, img_h = image.size
    inputs       = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits         = outputs.logits[0]
    pred_boxes_raw = outputs.pred_boxes[0]
    probs          = logits.softmax(-1)
    scores, pred_labels = probs[:, :-1].max(-1)
    keep = scores > CONF_THRESHOLD

    decoded_boxes = [
        center_to_corners(b[0], b[1], b[2], b[3], img_w, img_h)
        for b in pred_boxes_raw[keep].cpu().tolist()
    ]
    final_labels = pred_labels[keep].cpu().tolist()

    matched_gt = set()
    for p_box, p_lbl in zip(decoded_boxes, final_labels):
        best_iou = 0;  best_gt_idx = -1
        for gt_idx, gt in enumerate(gt_boxes):
            if gt_idx in matched_gt or p_lbl != gt["label"]:
                continue
            iou = compute_iou(p_box, gt["box"])
            if iou > best_iou:
                best_iou = iou;  best_gt_idx = gt_idx
        if best_iou >= IOU_THRESHOLD and best_gt_idx != -1:
            matched_gt.add(best_gt_idx)
            cid = gt_boxes[best_gt_idx]["label"]
            correct_detections  += 1
            running_iou         += best_iou
            class_hits[cid]     += 1       # hit for this specific class
            class_iou_sum[cid]  += best_iou

# ------------------------------------------------------------------------------
# 5. METRIC SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "=" * 52)
print("FINAL EVALUATION REPORT")
print("=" * 52)

if total_gt_objects > 0:
    overall_acc = (correct_detections / total_gt_objects) * 100
    mean_iou    = running_iou / correct_detections if correct_detections > 0 else 0.0

    print(f"\n{'OVERALL':}")
    print(f"  Total GT cells:      {total_gt_objects}")
    print(f"  Hits (IoU≥{IOU_THRESHOLD}):     {correct_detections}")
    print(f"  Accuracy:            {overall_acc:.2f}%")
    print(f"  Mean IoU:            {mean_iou:.4f}")

    print(f"\n{'CLASS-WISE BREAKDOWN':}")
    print(f"  {'Class':<22} {'GT':>5} {'Hits':>5} {'Acc':>8} {'mIoU':>8}")
    print(f"  {'-'*52}")
    for cid, name in sorted(id_to_class.items()):
        gt   = class_gt_counts[cid]
        hits = class_hits[cid]
        acc  = (hits / gt * 100) if gt > 0 else 0.0
        miou = (class_iou_sum[cid] / hits) if hits > 0 else 0.0
        print(f"  {name:<22} {gt:>5} {hits:>5} {acc:>7.2f}% {miou:>8.4f}")
else:
    print("Evaluation aborted. No matching ground-truth data found.")

Using device: cuda
Loaded 51 valid samples from /content/Random_Train_51/Random_Train_51/labels.json


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                                         | Status     |                                                                                       
----------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                       
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                       
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                       
model.backbone.conv_encoder.model.layer4.0.do


--- STARTING DETR FINE-TUNING ---


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- RUNNING EVALUATION ON TEST SPLIT ---

FINAL EVALUATION REPORT

OVERALL
  Total GT cells:      1061
  Hits (IoU≥0.5):     478
  Accuracy:            45.05%
  Mean IoU:            0.8102

CLASS-WISE BREAKDOWN
  Class                     GT  Hits      Acc     mIoU
  ----------------------------------------------------
  artefatto                 99     1    1.01%   0.7810
  emazia                     4     0    0.00%   0.0000
  eosinophil                53     0    0.00%   0.0000
  epithelial               498   477   95.78%   0.8102
  epithelial ciliated       11     0    0.00%   0.0000
  lymphocyte                12     0    0.00%   0.0000
  mast cell                  2     0    0.00%   0.0000
  metaplastic               21     0    0.00%   0.0000
  muciparous                47     0    0.00%   0.0000
  neutrophil               314     0    0.00%   0.0000
